In [1]:
import numpy as np
import numpy.linalg as LA
import matplotlib.pyplot as plt
from sympy import Symbol, exp, cos, sin, Matrix, diff, diag, expand, factor, latex, simplify

t = Symbol('t')

## Eigenvalue: Algebraic Multiplicity $3$ Geometric Multiplicity $2$.

Note that this is not possible with the problems we typically consider in our elementary differential equations course.  But we can create a first order linear system with this property.

### Example
$$
\begin{matrix}
\dot y_1 &=& -3 y_{1} + 2 y_{2} - 12 y_{3} + 7 y_{4}\\
\dot y_2 &=&  \phantom{0y_1+}y_{2} - y_{3} + y_{4}\\
\dot y_3 &=&  4 y_{1} - 2 y_{2} + 11 y_{3} - 5 y_{4}\\
\dot y_4 &=&  4 y_{1} - 2 y_{2} + 10 y_{3} - 4 y_{4}\end{matrix}
$$
or
$$
    \dot y = Ay = \begin{bmatrix}-3 & 2 & -12 & 7\\0 & 1 & -1 & 1\\4 & -2 & 11 & -5\\4 & -2 & 10 & -4\end{bmatrix}y
$$

In [2]:
A = Matrix([[-3, 2, -12, 7], [0, 1, -1, 1], [4, -2, 11, -5], [4, -2, 10, -4]])
A

Matrix([
[-3,  2, -12,  7],
[ 0,  1,  -1,  1],
[ 4, -2,  11, -5],
[ 4, -2,  10, -4]])

#### Characteristic polynomial

In [3]:
A.charpoly()

PurePoly(lambda**4 - 5*lambda**3 + 9*lambda**2 - 7*lambda + 2, lambda, domain='ZZ')

#### Characteristic polynomial, factored.

In [4]:
A.charpoly().as_expr().factor()

(lambda - 2)*(lambda - 1)**3

So $2$ is an eigenvalue of algebraic and geometric multiplicity one and $1$ is an eigenvalue of algebraic multiplicity $3$.  To see the geometric multiplicity of the eigenvalue $1$, we examine the null space of $A-I$.  The two small singular values suggest that the null space of $A-I$ has dimension $2$.

In [5]:
Af = np.array(A,dtype=float)
If = np.eye(4)
AmI = Af-If
LA.matrix_rank(AmI)

np.int64(2)

In [6]:
LA.svd(AmI,compute_uv=False)

array([2.24405919e+01, 1.19156767e+00, 5.18973576e-16, 1.97280127e-32])

### Eigenvalue 2 (the easy part)

First we find an eigenvector corresponding to $\lambda=2$.

In [7]:
λ,V = LA.eig(Af)
λ

array([1.        , 2.        , 0.99999998, 1.00000002])

In [8]:
v1 = V[:,1]
v1

array([ 5.77350269e-01,  7.35502242e-17, -5.77350269e-01, -5.77350269e-01])

In [9]:
v1/v1[0]

array([ 1.00000000e+00,  1.27392725e-16, -1.00000000e+00, -1.00000000e+00])

In [10]:
λ1 =2
w1 = np.array([1,0,-1,-1])
w1

array([ 1,  0, -1, -1])

In [11]:
A@w1 - λ1*w1

array([0, 0, 0, 0], dtype=object)

### Eigenvalue 1 (the hard part)
Recall the Caley-Hamilton theorem, that every matrix satisfies its characteristic polynomial.  In one form this is the identity
$$
    (A-I)^3(A-2I) = 0.
$$
We check:

In [12]:
I = Matrix(np.eye(4,dtype=int))
(A-I)@(A-I)@(A-I)@(A-2*I)

Matrix([
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]])

One way to interpret this is that the column space of $(A-2I)$, which must be three dimensional since its nullspace is one dimensional, is spanned by generalized eigenvectors corresponding to the eigenvalue $1$.

We also suspect that 
$$
    (A-I)(A-2I)\ne 0
$$
(else 1 would have geometric multiplicity 3) and
$$
    (A-I)^2(A-2I) = 0
$$
since there must be a vector in the nullspace of $(A-I)^2$ but not in the nullspace of $A-I$, hence the nullspace of $(A-I)^2$ would have to have dimension 3, thus making it the space spanned by the generalized eigenvectors of $A$ corresponding to $1$.  We check:

In [13]:
(A-I)@(A-2*I)

Matrix([
[0, 0,  8, -8],
[0, 0,  1, -1],
[0, 0, -6,  6],
[0, 0, -6,  6]])

In [14]:
(A-I)@(A-I)@(A-2*I)

Matrix([
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]])

A quick glance at $(A-I)(A-2)$ shows that its column space is spanned by its third column.  This must be an eigenvector  associated with $1$.

In [15]:
w2 = np.array([8,1,-6,-6])
(A-I)@w2

array([0, 0, 0, 0], dtype=object)

Furthermore, there must be a vector $w_3$ in the column space of $(A-2I)$ such that $(A-I)w_3 = w_2$.  Thus there must be some vector $x$ such that
$$
    (A-I)(A-2I)x = w_2
$$
and $w_3=(A-2I)x$.

In [16]:
Bf = np.array((A-I)@(A-2*I),dtype=float)
lss = LA.lstsq(Bf,w2)
lss

(array([ 0. ,  0. ,  0.5, -0.5]),
 array([], dtype=float64),
 np.int32(1),
 array([1.65529454e+01, 1.23527278e-15, 2.60545408e-32, 0.00000000e+00]))

In [17]:
w3 = (A-2*I)@lss[0]
w3

array([-9.50000000000000, -1.00000000000000, 7.00000000000000,
       8.00000000000000], dtype=object)

We turn to our standard tricks to get an all integer vector $w_3$.

In [18]:
w2 = 2*np.array([8,1,-6,-6])
lss = LA.lstsq(Bf,w2)
(A-2*I)@lss[0]

array([-19.0000000000000, -2.00000000000000, 14.0000000000000,
       16.0000000000000], dtype=object)

In [19]:
w3 = np.array([-19,-2,14,16])
A@w3 - w3 - w2

array([0, 0, 0, 0], dtype=object)

Finally, we need one more eigenvector of $A$ corresponding to $1$.  We will first try to find a vector in the nullspace of $A-I$ perpendicular to $w_2$.

In [20]:
_,s,vt = LA.svd(Af-If)
s

array([2.24405919e+01, 1.19156767e+00, 5.18973576e-16, 1.97280127e-32])

The nullspace of $A-I$ is approximately spanned by the last two columns of $V$:

In [21]:
v1,v2 = vt[-2:]
α = v1 @ w2
β = v2 @ w2
α,β

(np.float64(21.936833566256652), np.float64(8.171617531820436))

In [22]:
w2 - α * v1 - β *v2

array([ 3.10862447e-15,  9.76996262e-15, -8.85402862e-15, -1.36834988e-14])

In [23]:
w4 = -β * v1 + α * v2
w4@w2

np.float64(-2.1316282072803006e-14)

In [24]:
w4

array([ 4.71331991, 21.83011329,  4.96138938,  4.96138938])

In [25]:
w4/w4[0]

array([1.        , 4.63157895, 1.05263158, 1.05263158])

In [26]:
w4/(w4[2]-w4[0])

array([19., 88., 20., 20.])

In [27]:
w4 = np.array([19,88,20,20])
A@w4 - w4

array([0, 0, 0, 0], dtype=object)

### $J$
Now we are ready to form our matrix of generalized eigenvectors together with the Jordan canonical form for $A$.

In [28]:
P = Matrix(np.vstack((w1,w2,w3,w4)).T)
P

Matrix([
[ 1,  16, -19, 19],
[ 0,   2,  -2, 88],
[-1, -12,  14, 20],
[-1, -12,  16, 20]])

In [29]:
P.det()

548

In [30]:
J = diag(2,[[1,1],[0,1]],1)
J

Matrix([
[2, 0, 0, 0],
[0, 1, 1, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]])

In [31]:
A@P - P@J

Matrix([
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]])

## Fundamental Matrix Solution $e^{At}$.

In [32]:
expJt = diag(exp(2*t),[[exp(t),t*exp(t)],[0,exp(t)]],exp(t))
expJt

Matrix([
[exp(2*t),      0,        0,      0],
[       0, exp(t), t*exp(t),      0],
[       0,      0,   exp(t),      0],
[       0,      0,        0, exp(t)]])

In [33]:
expAt = P@expJt@P.inv()
expAt

Matrix([
[-4*exp(2*t) + 5*exp(t),  2*exp(2*t) - 2*exp(t), -8*t*exp(t) - 4*exp(2*t) + 4*exp(t),  8*t*exp(t) - exp(2*t) + exp(t)],
[                     0,                 exp(t),                           -t*exp(t),                        t*exp(t)],
[ 4*exp(2*t) - 4*exp(t), -2*exp(2*t) + 2*exp(t),  6*t*exp(t) + 4*exp(2*t) - 3*exp(t), -6*t*exp(t) + exp(2*t) - exp(t)],
[ 4*exp(2*t) - 4*exp(t), -2*exp(2*t) + 2*exp(t),  6*t*exp(t) + 4*exp(2*t) - 4*exp(t),          -6*t*exp(t) + exp(2*t)]])

In [34]:
simplify(expAt.det())

exp(5*t)

In [35]:
diff(expAt,t)-A@expAt

Matrix([
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]])